# Aegis OS - Session 3: Autonomous Landing
## The AI learns to descend and touch down softly on a landing pad

**Run each block ONE AT A TIME, top to bottom.**

### Before you start:
1. **Runtime > Change Runtime Type > T4 GPU**
2. Upload `session3_colab_package.zip` using the file browser (folder icon on the left)
3. Your `latest.pth` (Session 2 brain) is already in Google Drive - no need to re-upload!


## Block 1 - Unzip Your Package


In [ ]:
import zipfile, os

with zipfile.ZipFile('session3_colab_package.zip', 'r') as zf:
    zf.extractall('.')

for f in ['src/train_rl_session3.py', 'src/cognitive/rl_models.py']:
    print(('OK' if os.path.exists(f) else 'MISSING') + ': ' + f)

print('Block 1 complete!')


## Block 2 - Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted!')


## Block 3 - Verify Session 2 Brain is in Drive


In [ ]:
import torch, time, os

path = '/content/drive/MyDrive/aegis_checkpoints/latest/latest.pth'

if not os.path.exists(path):
    print('NOT FOUND: ' + path)
    print('Please upload latest.pth to: My Drive/aegis_checkpoints/latest/latest.pth')
else:
    c = torch.load(path, map_location='cpu', weights_only=False)
    print('Checkpoint found!')
    print('  Session:     ' + str(c.get('session')) + '  (should be 2)')
    print('  Update:      ' + str(c.get('update')))
    print('  Total Steps: ' + format(c.get('total_steps', 0), ','))
    print('  Best Reward: ' + str(round(c.get('best_reward', 0), 4)))
    dim = c['actor']['physics_net.0.weight'].shape[1]
    print('  Physics Dim: ' + str(dim) + '  (10 = Session 2 brain, ready for Session 3!)')
    print()
    print('Block 3 complete. Brain ready - no surgery needed for Session 3!')


## Block 4 - Check GPU


In [ ]:
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print('GPU Active: ' + name)
    print('VRAM: ' + str(vram) + ' GB')
    print('Ready to train!')
else:
    print('NO GPU! Go to Runtime > Change Runtime Type > T4 GPU')


## Block 5 - START SESSION 3 TRAINING

Runs for up to **4.5 hours**. Auto-saves every ~12 minutes to Drive.

You will see this confirming the Session 2 brain loaded successfully:
```
Loaded Session 2 brain into Session 3.
Prior steps: 1,572,864,000  |  Starting fresh update counter.
```
Then a live table showing **Landings/Env** climbing from 0.00 toward 0.5+ as the AI learns:
```
 Update |        Steps | Avg Reward | Landings/Env |     Loss
      0 |  ... |      -0.08 |         0.00 |  ...
    200 |  ... |      +1.20 |         0.12 |  ...
    800 |  ... |      +8.50 |         0.45 |  ...
```


In [ ]:
!python src/train_rl_session3.py \
    --session 3 \
    --checkpoint_dir /content/drive/MyDrive/aegis_checkpoints/ \
    --max_hours 5.5


## Block 6 - After Training: Save and Check Results


In [ ]:
import torch, time, os, shutil

latest_path = '/content/drive/MyDrive/aegis_checkpoints/latest/latest.pth'
final_path  = '/content/drive/MyDrive/aegis_checkpoints/session3_final.pth'

shutil.copy(latest_path, final_path)
print('Saved Session 3 brain as: ' + final_path)

c = torch.load(final_path, map_location='cpu', weights_only=False)
print()
print('=== SESSION 3 RESULTS ===')
print('Session:     ' + str(c.get('session')))
print('Update:      ' + str(c.get('update')) + ' / 1500')
print('Total Steps: ' + format(c.get('total_steps', 0), ','))
print('Best Reward: ' + str(round(c.get('best_reward', 0), 4)))
dim = c['actor']['physics_net.0.weight'].shape[1]
print('Physics Dim: ' + str(dim) + '  (10 = correct)')
saved = time.strftime('%Y-%m-%d %H:%M', time.localtime(c.get('timestamp', 0)))
print('Saved:       ' + saved)
print()
print('NEXT STEP: Download session3_final.pth from Google Drive')
print('           and save it to models/ folder on your computer')
